# Build a Classification Evaluation Helper

Once you have a trained classifier, a baseline, a tuned model, or anything in between; the next question is the same: is it actually good, and what should we do with it? Metrics like accuracy or ROC AUC don't answer that on their own.



## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn matplotlib python-dotenv

In [14]:
import inspect
import warnings
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))
sys.path.append(str(Path("../03-Modeling-Helpers").resolve()))

from preprocessing_pipeline import preprocessing_pipeline
from classification_helper import classification_helper

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


GEN_CONFIG = {"temperature": 0.0, "seed": 42}
pd.set_option("display.max_colwidth", None)

warnings.filterwarnings("ignore")

## 2 - Load Data and Run Model

Keep the opening consistent with Lesson 3.1: import the preprocessing pipeline, build the train/test split from the raw HR dataset, then run the baseline classification helper to get predictions.

In [15]:
result = preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="Attrition",
    task="classification",
)
X_train = result.X_train_enc
X_test = result.X_test_enc
y_train = result.y_train
y_test = result.y_test

results = classification_helper(X_train, X_test, y_train, y_test)

Loaded: (5030, 25)
Split raw: train=(4024, 25) test=(1006, 25)
Cleaned: train=(4003, 25) missing=0 | test=(1006, 25) missing=0
Encoded: train=(4003, 49) test=(1006, 49)


In [16]:
print(f"Model: {results['model_name']}")


Model: sklearn.linear_model.LogisticRegression


## 3 - Compute Metrics

We compute classification metrics with scikit-learn and pass them to the LLM, which interprets the results, flags anomalies like class imbalance and weak recall, generates an evaluation report and recommends an optimal decision threshold.

In [17]:
def compute_classification_metrics(y_true, y_pred, y_proba=None, threshold=None):
    """Compute standard classification metrics and optionally re-score at a custom threshold."""
    y_true = pd.Series(y_true).astype(int)
    used_threshold = threshold if threshold is not None else 0.50

    if threshold is not None and y_proba is not None:
        y_pred_eval = (pd.Series(y_proba) >= threshold).astype(int)
    else:
        y_pred_eval = pd.Series(y_pred).astype(int)

    metrics = {
        "threshold": float(used_threshold),
        "accuracy": float(accuracy_score(y_true, y_pred_eval)),
        "precision": float(precision_score(y_true, y_pred_eval, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred_eval, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred_eval, zero_division=0)),
        "class_ratio": float(y_true.mean()),
        "predicted_positive_rate": float(pd.Series(y_pred_eval).mean()),
        "confusion_matrix": confusion_matrix(y_true, y_pred_eval).tolist(),
    }

    if y_proba is not None and y_true.nunique() > 1:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba))
    else:
        metrics["roc_auc"] = None

    return metrics


In [18]:
metrics = compute_classification_metrics(
    y_test,
    results["y_pred"],
    results["y_proba"],
)

pd.DataFrame([metrics]).T.rename(columns={0: "value"}).round(4)

,value
threshold,0.5
accuracy,0.696819
precision,0.166172
recall,0.7
f1,0.268585
class_ratio,0.079523
predicted_positive_rate,0.33499
confusion_matrix,"[[645, 281], [24, 56]]"
roc_auc,0.739619


The model can look acceptable on accuracy while still being weak on precision or recall. 

## 4 - LLM Flags Anomalies

Ask the LLM to look for red flags. This is where it should catch patterns like high accuracy masking weak recall on an imbalanced target.

In [19]:
FLAG_PROMPT = (
    "You are a senior data scientist auditing a classification model. "
    "Given a metrics dict, identify any red flags or anomalies. "
    "Common issues: high accuracy but low recall on imbalanced data, precision-recall tradeoff, weak F1, ROC-AUC close to 0.5. "
    "Return ONLY valid JSON: a list of objects with keys flag (string), severity (high/medium/low), explanation (string). "
    "No markdown fences. No text outside the JSON."
)


def flag_anomalies(metrics):

    content = f"Metrics: {json.dumps(metrics, indent=2)}"
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=content,
        config={"system_instruction": FLAG_PROMPT, **GEN_CONFIG},
    )
    text = (resp.text or "").strip()
    text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    return json.loads(text)


In [20]:
flags = flag_anomalies(metrics)
flags_df = pd.DataFrame(flags)
flags_df


,flag,severity,explanation
0,High Class Imbalance,medium,"The positive class constitutes only about 8% of the dataset (class_ratio: 0.0795), indicating a significant class imbalance. This makes accuracy a misleading metric and necessitates a focus on precision, recall, and F1-score for the minority class."
1,Very Low Precision for Positive Class,high,"The precision score of 0.166 is very low. This means that when the model predicts a positive outcome, it is only correct about 16.6% of the time. A high number of false positives (281) are being generated, which could lead to wasted resources or incorrect actions depending on the application's cost of false positives."
2,Poor F1-Score Despite Good Recall,medium,"While the recall for the positive class is good (0.70), the F1-score is low (0.268). This indicates that the model's overall effectiveness for the positive class is poor due to the extremely low precision, suggesting an imbalance in the precision-recall tradeoff."
3,Precision-Recall Tradeoff Imbalance,medium,"The model exhibits a high recall (0.70) but very low precision (0.166). This suggests the model might be overly aggressive in identifying positive cases, leading to many false positives. It's worth investigating if adjusting the classification threshold (currently 0.5) could improve precision without an unacceptable drop in recall, depending on the business objective."


## 5 - Save for Reuse

If you want to reuse these LLM evaluation helpers later, package the core functions into a Python file.

In [22]:
components = [
    "import inspect",
    "import json",
    "import os",
    "import re",
    "from pathlib import Path",
    "",
    "import pandas as pd",
    "from dotenv import load_dotenv",
    "from google import genai",
    "from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score",
    "",
    "PROJECT_ROOT = Path.cwd().resolve()",
    "load_dotenv(PROJECT_ROOT / '.env')",
    "api_key = os.getenv('GEMINI_API_KEY')",
    "client = genai.Client(api_key=api_key) if api_key else None",
    f"GEN_CONFIG = {repr(GEN_CONFIG)}",
    f"FLAG_PROMPT = {repr(FLAG_PROMPT)}",
    "",
    inspect.getsource(compute_classification_metrics),
    "",
    inspect.getsource(flag_anomalies),
]

with open("classification_eval_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved classification_eval_helper.py")

Saved classification_eval_helper.py
